# 01 — Data Preparation

**Steps:**
1. Resolve file UUIDs → patient barcodes via the GDC API (`scripts/ingest_gdc.resolve_barcodes`).
2. Dedup aliquots → one row per patient.
3. Join survival labels from TCGA-CDR.
4. Stack RNA-seq TSVs into a (patients × genes) matrix.
5. Filter genes, apply `log2(FPKM_UQ + 1)`, standardise per gene.
6. Write the cleaned cohort table to `data/processed/`.

## Cohort and paths

In [1]:
import pandas as pd
from pathlib import Path

COHORT = "LUSC"
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — File → barcode resolution

The TCGA RNA-seq TSVs don't carry the patient barcode in the file body. We query the GDC `/files` endpoint for every file UUID in the manifest and pull back the nested aliquot barcode. See `scripts/ingest_gdc.resolve_barcodes`, which writes `data/raw/{COHORT}_barcodes.csv` with one row per file:

| column | meaning |
|---|---|
| `file_id` | GDC file UUID — matches the folder name in `data/raw/{COHORT}/` |
| `patient_barcode` | `TCGA-XX-XXXX` — join key to TCGA-CDR |
| `sample_barcode` | `TCGA-XX-XXXX-01A` — adds sample-type + vial |
| `aliquot_barcode` | full 28-char barcode — the canonical id for the row |
| `sample_type` | should all be `Primary Tumor` (we filtered the manifest on this) |
| `tss` | tissue source site code (positions 2-3 of the patient barcode) — the institution |

In [2]:
barcodes = pd.read_csv(RAW_DIR / f"{COHORT}_barcodes.csv")
print("shape:", barcodes.shape)
barcodes.head()

shape: (511, 8)


,file_id,file_name,project_id,patient_barcode,sample_barcode,aliquot_barcode,sample_type,tss
0,ada4ae4b-1959-49a1-b3db-59675ba2ade0,1579c2d5-2873-473b-b299-19100bac3e9f.rna_seq.a...,TCGA-LUSC,TCGA-96-A4JK,TCGA-96-A4JK-01A,TCGA-96-A4JK-01A-11R-A24Z-07,Primary Tumor,96
1,a2541c72-d02f-4acb-8d35-537080c94ea8,61192f9b-94eb-49f4-b2b5-eafe75092930.rna_seq.a...,TCGA-LUSC,TCGA-43-3394,TCGA-43-3394-01A,TCGA-43-3394-01A-01R-0980-07,Primary Tumor,43
2,343f1771-4a3d-4a03-9eb0-319dada0b373,042d1918-78a0-4655-a421-0c8b38538170.rna_seq.a...,TCGA-LUSC,TCGA-85-A4QQ,TCGA-85-A4QQ-01A,TCGA-85-A4QQ-01A-41R-A262-07,Primary Tumor,85
3,aa3501a9-217b-4d3a-8dab-0fb359728726,b039b20f-5532-41b4-a92e-b48bc87f89c1.rna_seq.a...,TCGA-LUSC,TCGA-18-3419,TCGA-18-3419-01A,TCGA-18-3419-01A-01R-0980-07,Primary Tumor,18
4,17ddff0c-a5c2-4f82-b638-fe13c20de1d3,385d5ffe-02ae-4fed-8590-b332d08597bb.rna_seq.a...,TCGA-LUSC,TCGA-O2-A5IB,TCGA-O2-A5IB-01A,TCGA-O2-A5IB-01A-11R-A27Q-07,Primary Tumor,O2


In [ ]:
assert (barcodes["sample_type"] == "Primary Tumor").all()
assert (barcodes["project_id"] == f"TCGA-{COHORT}").all()

## Step 2 — Dedup aliquots

Some patients have >1 aliquot sequenced (different vials, repeat extractions, multiple sequencing centres). Each aliquot is a separate file in GDC but the survival label is patient-level — so duplicate aliquots would upweight a non-random subset of patients in the likelihood.

**Rule:** keep the alphabetically first `aliquot_barcode` per patient. Deterministic, reproducible, no dependence on the raw TSVs.

For LUSC we expect ~10 affected patients out of 501.

In [ ]:
files_per_patient = barcodes.groupby("patient_barcode").size().value_counts().sort_index()
print("files per patient (before dedup):")
print(files_per_patient)

dedup = (
    barcodes.sort_values("aliquot_barcode")
            .groupby("patient_barcode", as_index=False)
            .first()
)

print(f"\nbefore: {len(barcodes)} files")
print(f"after:  {len(dedup)} patients")
print(f"dropped {len(barcodes) - len(dedup)} duplicate aliquots")

### TSS distribution after dedup

TSS is the random-effect grouping variable in Model B. Sites with very few patients will have weakly informed random intercepts; the long tail is worth eyeballing before fitting.

In [ ]:
tss_counts = dedup["tss"].value_counts()
print(f"distinct TSS: {len(tss_counts)}")
print()
print(tss_counts.describe())
print()
print("bottom 10 (smallest sites):")
print(tss_counts.tail(10))

## Step 3 — Join TCGA-CDR survival labels

Source: `data/raw/tcga_cdr.xlsx`, sheet `TCGA-CDR` (Liu et al. 2018). Join on `bcr_patient_barcode == patient_barcode`. Endpoint: OS (overall survival). We need `OS` (event indicator) and `OS.time` (days). Patients missing either are dropped.

*Pending — fill in below.*

In [ ]:
# TODO: load TCGA-CDR sheet, filter to type == COHORT, merge on patient_barcode,
# drop rows with null OS / OS.time, report counts at each step.

## Step 4 — Build the gene matrix

For each surviving patient, read `data/raw/{COHORT}/{file_id}/*.rna_seq.augmented_star_gene_counts.tsv`, drop the 4 STAR QC rows at the top (`N_unmapped`, `N_multimapping`, `N_noFeature`, `N_ambiguous`), and take `fpkm_uq_unstranded` keyed by `gene_id`. Verify gene order is identical across files, then stack into a (patients × genes) matrix.

*Pending — requires the download to finish.*

In [ ]:
# TODO: stack per-patient TSVs into a (patients × genes) FPKM-UQ matrix.

## Step 5 — Gene filter, transform, standardise

- Filter to `gene_type == "protein_coding"`.
- (Modelling choice, deferred: subset to MSigDB hallmark gene sets — ~671 genes — vs top-variance selection.)
- Apply `log2(FPKM_UQ + 1)`.
- Standardise each gene to mean 0, std 1 across patients **within this cohort**.

*Pending.*

In [ ]:
# TODO: protein-coding filter, log2(FPKM+1), per-gene standardisation.

## Step 6 — Write the cleaned cohort table

Final output: `data/processed/{COHORT}.parquet` with one row per patient. Columns: `patient_barcode, tss, OS, OS_time, <gene_1>, ..., <gene_N>`.

*Pending.*

In [ ]:
# TODO: write data/processed/{COHORT}.parquet